# PostgreSQL vs Databricks Multi-SQL Check Notebook

Paste checks in this format:

```text
<=>Check Name<=>
#PG#
SELECT ...
#DB#
SELECT ...
<=>End<=>
```

The notebook parses each check, executes the PostgreSQL and Databricks SQL, compares complete result sets, ignores row order, preserves duplicate-row counts, and shows `MATCH`, `MISMATCH`, or `ERROR`.


In [ ]:
# CONFIGURATION

POSTGRES_CONFIG = {
    "host": "your-postgres-host",
    "port": 5432,
    "database": "your_database",
    "user": "your_username",
    "password": "your_password"
}


In [ ]:
# PASTE ALL SQL CHECKS HERE

SQL_CHECKS = r"""
<=>Claim Count<=>
#PG#
SELECT COUNT(*) AS cnt
FROM claims.claim_details

#DB#
SELECT COUNT(*) AS cnt
FROM catalog.schema.claim_details
<=>End<=>

<=>Status Summary<=>
#PG#
SELECT status, COUNT(*) AS cnt
FROM claims.claim_details
GROUP BY status

#DB#
SELECT status, COUNT(*) AS cnt
FROM catalog.schema.claim_details
GROUP BY status
<=>End<=>
"""


In [ ]:
# IMPORTS

import re
from decimal import Decimal

import pandas as pd
import psycopg2
from pyspark.sql import functions as F


In [ ]:
# PARSE CHECK BLOCKS

def parse_sql_checks(text):
    pattern = re.compile(
        r"<=>\s*(.*?)\s*<=>\s*"
        r"#PG#\s*(.*?)\s*"
        r"#DB#\s*(.*?)\s*"
        r"<=>End<=>",
        re.IGNORECASE | re.DOTALL
    )

    checks = []

    for match in pattern.finditer(text):
        checks.append({
            "check_name": match.group(1).strip(),
            "pg_sql": match.group(2).strip(),
            "db_sql": match.group(3).strip()
        })

    return checks


checks = parse_sql_checks(SQL_CHECKS)

print(f"Checks found: {len(checks)}")

for check in checks:
    print("-", check["check_name"])


In [ ]:
# QUERY EXECUTION

def run_postgres_query(sql_text):
    conn = psycopg2.connect(
        host=POSTGRES_CONFIG["host"],
        port=POSTGRES_CONFIG["port"],
        database=POSTGRES_CONFIG["database"],
        user=POSTGRES_CONFIG["user"],
        password=POSTGRES_CONFIG["password"]
    )

    try:
        return pd.read_sql_query(sql_text, conn)
    finally:
        conn.close()


def run_databricks_query(sql_text):
    df = spark.sql(sql_text)

    select_exprs = []

    for field in df.schema.fields:
        if field.dataType.typeName() in {"timestamp", "timestamp_ntz", "date"}:
            select_exprs.append(
                F.col(field.name).cast("string").alias(field.name)
            )
        else:
            select_exprs.append(F.col(field.name))

    return df.select(*select_exprs).toPandas()


In [ ]:
# RESULT SET COMPARISON

def normalize_value(value):
    if value is None:
        return "<NULL>"

    try:
        if pd.isna(value):
            return "<NULL>"
    except Exception:
        pass

    if isinstance(value, Decimal):
        return format(value, "f")

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if hasattr(value, "isoformat"):
        try:
            return value.isoformat()
        except Exception:
            pass

    if isinstance(value, bool):
        return "true" if value else "false"

    return str(value).strip()


def normalize_dataframe(df):
    out = df.copy()

    out.columns = [
        str(c).strip().lower()
        for c in out.columns
    ]

    for col in out.columns:
        out[col] = out[col].map(normalize_value)

    return out


def compare_result_sets(pg_df, db_df):
    pg = normalize_dataframe(pg_df)
    db = normalize_dataframe(db_df)

    if set(pg.columns) != set(db.columns):
        return {
            "status": "MISMATCH",
            "reason": "COLUMN_MISMATCH",
            "pg_rows": len(pg),
            "db_rows": len(db),
            "mismatch_rows": None,
            "details": pd.DataFrame()
        }

    columns = sorted(pg.columns)

    pg = pg[columns]
    db = db[columns]

    pg_counts = (
        pg.groupby(columns, dropna=False)
          .size()
          .reset_index(name="pg_count")
    )

    db_counts = (
        db.groupby(columns, dropna=False)
          .size()
          .reset_index(name="db_count")
    )

    comparison = pd.merge(
        pg_counts,
        db_counts,
        on=columns,
        how="outer"
    )

    comparison["pg_count"] = comparison["pg_count"].fillna(0).astype(int)
    comparison["db_count"] = comparison["db_count"].fillna(0).astype(int)

    mismatch = comparison[
        comparison["pg_count"] != comparison["db_count"]
    ].copy()

    if mismatch.empty:
        return {
            "status": "MATCH",
            "reason": "",
            "pg_rows": len(pg),
            "db_rows": len(db),
            "mismatch_rows": 0,
            "details": pd.DataFrame()
        }

    return {
        "status": "MISMATCH",
        "reason": "RESULT_SET_MISMATCH",
        "pg_rows": len(pg),
        "db_rows": len(db),
        "mismatch_rows": len(mismatch),
        "details": mismatch
    }


In [ ]:
# EXECUTE ALL CHECKS

summary_rows = []
mismatch_details = {}

for check in checks:
    check_name = check["check_name"]

    print("\n" + "=" * 80)
    print(f"Running: {check_name}")
    print("=" * 80)

    try:
        pg_df = run_postgres_query(check["pg_sql"])
        db_df = run_databricks_query(check["db_sql"])

        result = compare_result_sets(pg_df, db_df)

        summary_rows.append({
            "check_name": check_name,
            "status": result["status"],
            "reason": result.get("reason", ""),
            "postgres_rows": result.get("pg_rows"),
            "databricks_rows": result.get("db_rows"),
            "mismatch_patterns": result.get("mismatch_rows")
        })

        print(
            f"{check_name}: {result['status']} | "
            f"PG rows={result.get('pg_rows')} | "
            f"DB rows={result.get('db_rows')}"
        )

        if (
            result["status"] == "MISMATCH"
            and result.get("details") is not None
            and not result["details"].empty
        ):
            mismatch_details[check_name] = result["details"]

    except Exception as exc:
        summary_rows.append({
            "check_name": check_name,
            "status": "ERROR",
            "reason": str(exc),
            "postgres_rows": None,
            "databricks_rows": None,
            "mismatch_patterns": None
        })

        print(f"{check_name}: ERROR -> {exc}")


summary_df = pd.DataFrame(summary_rows)

print("\nFINAL SUMMARY")
display(
    spark.createDataFrame(
        summary_df.astype(str)
    )
)


In [ ]:
# OPTIONAL - DISPLAY MISMATCH DETAILS

for check_name, detail_df in mismatch_details.items():
    print("\n" + "=" * 80)
    print(f"MISMATCH DETAILS: {check_name}")
    print("=" * 80)

    display(
        spark.createDataFrame(
            detail_df.astype(str)
        )
    )
